# 06 — Fairness Analysis

Evaluate whether the model performs equally across demographic groups.
This is critical for healthcare AI — a model that works well on average
but poorly for certain groups (age, sex, BMI) is dangerous.

**Key Question:** Does our model discriminate against any patient subgroup?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

## 1. Load Models & Data

In [ ]:
from src.model import load_model
from src.fairness import (
    evaluate_by_group, create_age_groups, create_bmi_groups,
    compute_fairness_gap, generate_fairness_report,
)
from src.preprocessing import RISK_LABEL_MAP

processed = Path('../data/processed')
outputs = Path('../outputs')

# Load test data (unscaled for grouping, scaled for prediction)
X_test_d = pd.read_csv(processed / 'diabetes_X_test.csv')
y_test_d = pd.read_csv(processed / 'diabetes_y_test.csv').squeeze()
X_test_h = pd.read_csv(processed / 'heart_X_test.csv')
y_test_h = pd.read_csv(processed / 'heart_y_test.csv').squeeze()

# Load best model
model_d = load_model('xgboost', 'diabetes')
model_h = load_model('xgboost', 'heart')
print('Models and data loaded')

## 2. Diabetes — Fairness by Age Group

In [ ]:
# Create age groups from the scaled Age feature
# Note: we need to use the scaled values since the model was trained on scaled data
X_d_eval = X_test_d.copy()
X_d_eval['Age_Group'] = create_age_groups(
    # Use the raw-ish age column (index 7 in feature list)
    pd.Series(X_test_d.iloc[:, 7], name='Age')
)

age_metrics_d = evaluate_by_group(model_d, X_d_eval, y_test_d, 'Age_Group')
print('Diabetes — Performance by Age Group:')
print(age_metrics_d.to_string(index=False))

In [ ]:
# Visualize
fig = px.bar(
    age_metrics_d, x='Group', y=['F1-Macro', 'Precision', 'Recall'],
    barmode='group', title='Diabetes — Model Performance by Age Group',
    color_discrete_sequence=['#4e79a7', '#f28e2b', '#e15759'],
)
fig.update_layout(yaxis_range=[0, 1], yaxis_title='Score')
fig.show()

In [ ]:
# Fairness gap
age_gap_d = compute_fairness_gap(age_metrics_d)
print(f'\nFairness Gap (Age):')
print(f'  Best group:  {age_gap_d["best_group"]} ({age_gap_d["best_score"]:.4f})')
print(f'  Worst group: {age_gap_d["worst_group"]} ({age_gap_d["worst_score"]:.4f})')
print(f'  Gap:         {age_gap_d["gap"]:.4f}')
print(f'  Fair (gap <= 0.10): {age_gap_d["is_fair"]}')

## 3. Heart Disease — Fairness by Age Group

In [ ]:
X_h_eval = X_test_h.copy()
X_h_eval['Age_Group'] = create_age_groups(
    pd.Series(X_test_h.iloc[:, 0], name='age')
)

age_metrics_h = evaluate_by_group(model_h, X_h_eval, y_test_h, 'Age_Group')
print('Heart Disease — Performance by Age Group:')
print(age_metrics_h.to_string(index=False))

In [ ]:
fig = px.bar(
    age_metrics_h, x='Group', y=['F1-Macro', 'Precision', 'Recall'],
    barmode='group', title='Heart Disease — Model Performance by Age Group',
    color_discrete_sequence=['#4e79a7', '#f28e2b', '#e15759'],
)
fig.update_layout(yaxis_range=[0, 1], yaxis_title='Score')
fig.show()

## 4. Heart Disease — Fairness by Sex

In [ ]:
sex_col = 'sex' if 'sex' in X_test_h.columns else X_test_h.columns[1]
sex_metrics = evaluate_by_group(
    model_h, X_test_h, y_test_h, sex_col,
    group_labels={0: 'Female', 1: 'Male', 0.0: 'Female', 1.0: 'Male'},
)
print('Heart Disease — Performance by Sex:')
print(sex_metrics.to_string(index=False))

sex_gap = compute_fairness_gap(sex_metrics)
print(f'\nFairness Gap (Sex):')
print(f'  Gap: {sex_gap["gap"]:.4f} — Fair: {sex_gap["is_fair"]}')

In [ ]:
fig = px.bar(
    sex_metrics, x='Group', y=['F1-Macro', 'Precision', 'Recall'],
    barmode='group', title='Heart Disease — Model Performance by Sex',
    color_discrete_sequence=['#4e79a7', '#f28e2b', '#e15759'],
)
fig.update_layout(yaxis_range=[0, 1], yaxis_title='Score')
fig.show()

## 5. Complete Fairness Report

In [ ]:
# Generate full reports
report_d = generate_fairness_report(model_d, X_test_d, y_test_d, 'diabetes')
report_h = generate_fairness_report(model_h, X_test_h, y_test_h, 'heart')

print('=== DIABETES FAIRNESS REPORT ===')
print('\nAge Groups:')
print(report_d['age_groups'].to_string(index=False))
print(f'Age Gap: {report_d["age_fairness"]["gap"]:.4f} (Fair: {report_d["age_fairness"]["is_fair"]})')

if 'bmi_groups' in report_d:
    print('\nBMI Groups:')
    print(report_d['bmi_groups'].to_string(index=False))
    print(f'BMI Gap: {report_d["bmi_fairness"]["gap"]:.4f} (Fair: {report_d["bmi_fairness"]["is_fair"]})')

print('\n=== HEART DISEASE FAIRNESS REPORT ===')
print('\nAge Groups:')
print(report_h['age_groups'].to_string(index=False))
print(f'Age Gap: {report_h["age_fairness"]["gap"]:.4f} (Fair: {report_h["age_fairness"]["is_fair"]})')

if 'sex_groups' in report_h:
    print('\nSex Groups:')
    print(report_h['sex_groups'].to_string(index=False))
    print(f'Sex Gap: {report_h["sex_fairness"]["gap"]:.4f} (Fair: {report_h["sex_fairness"]["is_fair"]})')

## 6. Fairness Summary Visualization

In [ ]:
# Combined fairness gap chart
gaps = []
gaps.append({'Dataset': 'Diabetes', 'Group Type': 'Age', 'Gap': report_d['age_fairness']['gap']})
if 'bmi_fairness' in report_d:
    gaps.append({'Dataset': 'Diabetes', 'Group Type': 'BMI', 'Gap': report_d['bmi_fairness']['gap']})
gaps.append({'Dataset': 'Heart', 'Group Type': 'Age', 'Gap': report_h['age_fairness']['gap']})
if 'sex_fairness' in report_h:
    gaps.append({'Dataset': 'Heart', 'Group Type': 'Sex', 'Gap': report_h['sex_fairness']['gap']})

gaps_df = pd.DataFrame(gaps)

fig = px.bar(
    gaps_df, x='Group Type', y='Gap', color='Dataset',
    barmode='group', title='Fairness Gaps Across Demographic Groups',
    color_discrete_sequence=['#4e79a7', '#e15759'],
)
fig.add_hline(y=0.10, line_dash='dash', line_color='red',
              annotation_text='Fairness threshold (0.10)')
fig.update_layout(yaxis_title='F1-Macro Gap', yaxis_range=[0, max(0.3, gaps_df['Gap'].max() + 0.05)])
fig.show()

# Save
gaps_df.to_csv(outputs / 'fairness_gaps.csv', index=False)
print('Fairness results saved to outputs/fairness_gaps.csv')

## 7. Key Findings & Recommendations

### Findings
- ...
- ...

### Recommendations
1. If any group has a fairness gap > 0.10, investigate whether the training data is underrepresented for that group
2. Consider collecting more data for underperforming groups
3. Explore techniques like reweighting or adversarial debiasing if gaps persist
4. **For Bangladesh deployment:** Must retrain on local demographic data — these datasets are NOT representative